# 🧠 Forex_DNN Machine Learning Model Training Workbench

This interactive notebook is designed to train, evaluate, and inspect machine learning models in the Forex_DNN framework.

### Key Features Included:
1. **Execution Modes**:
   - `DEBUG`: Trains a fast, lightweight classifier on a tiny synthetic dataset (500 samples), saving the model to a local debug folder without modifying production files.
   - `FULL`: Runs the standard, full production-grade model training pipeline using all processed dataset samples, saving models directly to workspace directories.
2. **Feature Importances**: Highlights and plots the top 15 predictive indicators.
3. **Performance Diagnostics**: Generates a standard confusion matrix, classification reports, and cross-validation summaries.

In [ ]:
import os
import sys

# Ensure we can import from the framework root
framework_root = os.path.abspath(os.path.join(os.getcwd(), '../..'))
if framework_root not in sys.path:
    sys.path.append(framework_root)
print(f"Framework root added to sys.path: {framework_root}")

In [ ]:
import argparse
import json
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score
)

from Configs.path_manager import PathManager
from ML.models.market_state_classifier import MarketStateClassifier
from ML.trainer import Trainer
from ML.evaluator import Evaluator
from ML.feature_registry import FeatureRegistry

## ⚙️ 1. PARAMETERS & INPUT CONFIGURATION

In [ ]:
# Choose "DEBUG" or "FULL"
MODE = "DEBUG" 

# Model input and output paths based on execution mode
if MODE == "DEBUG":
    DATASET_PATH = os.path.join(PathManager.get_path("temporary"), "debug_market_state_dataset.parquet")
    MODEL_SAVE_PATH = os.path.join(PathManager.get_path("temporary"), "debug_market_state_classifier.joblib")
else:
    DATASET_PATH = PathManager.get_relative_path("datasets", "market_state_dataset.parquet")
    MODEL_SAVE_PATH = PathManager.get_relative_path("models", "MarketState/market_state_classifier.joblib")

RANDOM_SEED = 42

## 💾 2. DATASET PRE-SEEDING & PREPARATION

In [ ]:
np.random.seed(RANDOM_SEED)

# Check if dataset exists, if not generate or use synthetic dummy data for fast setup
if not os.path.exists(DATASET_PATH):
    print(f"Dataset path '{DATASET_PATH}' not found. Generating a high-quality synthetic dataset for fast setup...")
    n_samples = 500
    reg = FeatureRegistry(load_defaults=True)
    enabled_features = [f.name for f in reg.list_enabled()]
    
    # Make sure we have some indicator fields
    data = {feat: np.random.randn(n_samples) for feat in enabled_features if feat not in ["Datetime", "timestamp"]}
    df_dummy = pd.DataFrame(data)
    
    # Add categorical metadata and targets
    df_dummy["target"] = np.random.choice(["TREND", "RANGE", "TRANSITION"], n_samples)
    df_dummy["timestamp"] = pd.date_range("2026-01-01", periods=n_samples, freq="5min")
    
    os.makedirs(os.path.dirname(os.path.abspath(DATASET_PATH)), exist_ok=True)
    df_dummy.to_parquet(DATASET_PATH, index=False)
    print(f"Synthetic dataset pre-seeded successfully at {DATASET_PATH}")

# Load dataset
df = pd.read_parquet(DATASET_PATH)
print(f"Successfully loaded dataset. Shape: {df.shape}")

## 🦾 3. MODEL TRAINING & CALIBRATION

In [ ]:
PathManager.ensure_all_dirs()

# Initialize model
config_path = PathManager.get_relative_path("config", "market_state.yaml") if os.path.exists(PathManager.get_relative_path("config", "market_state.yaml")) else None

clf = MarketStateClassifier(
    model_type="lightgbm",
    config_path=config_path,
    random_state=RANDOM_SEED
)

# Configure training targets and feature columns
target_col = "target" if "target" in df.columns else "label"

metadata_cols = [
    "label", "target", "confidence", "timestamp", "Datetime", "symbol", "timeframe",
    "window_start", "window_end", "sample_id", "label_version", "engine_version",
    "meta_labeler_rule_fired", "Open", "High", "Low", "Close", "TickVolume", "Spread",
    "ema_50", "ema_600", "ema_800"
]
feature_cols = [c for c in df.columns if c not in metadata_cols and not c.startswith("meta_labeler_")]

# Execute Trainer split and train
trainer = Trainer(random_seed=RANDOM_SEED)
train_results = trainer.train_model(
    model=clf,
    df=df,
    target_col=target_col,
    feature_cols=feature_cols,
    test_size=0.2,
    chronological=True,
    dataset_version="debug_v1.0" if MODE == "DEBUG" else "production_v1.0",
    dataset_hash="debug_hash_xyz",
    model_save_path=MODEL_SAVE_PATH,
    is_production=(MODE == "FULL"),
    version="1.0.0"
)

print("\n--- Training successfully completed! ---")
X_val = train_results["X_val"]
y_val = train_results["y_val"]

## 📊 4. TRAINING EVALUATION & DIAGNOSTICS

In [ ]:
# 1. Feature Importances
if hasattr(clf.model, "feature_importances_"):
    importances = clf.model.feature_importances_
    indices = np.argsort(importances)[::-1][:15]
    
    plt.figure(figsize=(12, 6))
    sns.barplot(x=importances[indices], y=np.array(feature_cols)[indices], palette="viridis")
    plt.title("Top 15 Predictive Features by LightGBM Importance")
    plt.xlabel("Feature Importance Score")
    plt.ylabel("Feature Indicator Name")
    plt.grid(True, linestyle=":", alpha=0.5)
    plt.tight_layout()
    plt.show()

# 2. Validation Metrics: Confusion Matrix & Classification Report
y_pred = clf.model.predict(X_val)

print("\n=== Classification Report ===")
print(classification_report(y_val, y_pred))

cm = confusion_matrix(y_val, y_pred)
unique_classes = np.unique(y_val)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=unique_classes, yticklabels=unique_classes)
plt.title("Classification Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show()